# Iris decision lab
Adapted from [Aurélien Géron's handson-ml3, chapter 6](https://github.com/ageron/handson-ml3/blob/e707c2d659abafb9b1f9fd927907619a128db8d7/06_decision_trees.ipynb), licensed under Apache-2.0. The original notebook, license and provenance remain unchanged in `source/`.

The original Iris example fits a depth-2 decision tree to petal length and width. This adaptation uses **all four measurements**, compares a decision tree, a random forest and logistic regression, and reserves 30% of the flowers for evaluation. Only the relevant example was read; the upstream notebook is not executed.

These stages use the project path already provided by AGILAB (or the notebook runner) to import `models`. No working-directory assumptions, downloads or environment setup are needed. The metrics artifact is written to the execution working directory.


## 1. Import and configure
The shared module exposes reusable split, training and evaluation stages. Change depth or seed here for an exploratory run.


In [ ]:
import json
from pathlib import Path

from models import build_models, load_split, train_models, evaluate_models

max_depth = 3
seed = 42


## 2. Train
Every estimator sees the same training rows. The logistic regression pipeline fits its scaler on those rows only; no test data enter training. Depth controls both the tree and forest.


In [ ]:
iris, X_train, X_test, y_train, y_test = load_split(seed=seed)
models = train_models(X_train, y_train, max_depth=max_depth, seed=seed)
print(f"Training: {len(y_train)} flowers; held out: {len(y_test)} flowers")


## 3. Evaluate and export
Accuracy ranks models on this one held-out split. Training scores help reveal a potential generalization gap. Preserve tied leaders rather than choosing arbitrarily.


In [ ]:
comparison, predictions = evaluate_models(models, X_train, X_test, y_train, y_test)
metrics = [
    {"model": row["model"], "accuracy": float(row["test_accuracy"])}
    for row in comparison.to_dict(orient="records")
]
Path("metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False) + "\n", encoding="utf-8")
best_accuracy = comparison["test_accuracy"].max()
leaders = comparison.loc[comparison["test_accuracy"] == best_accuracy, "model"].tolist()
print(comparison.to_string(index=False))
print("Leader(s) on this split:", ", ".join(leaders))
print(f"Best held-out accuracy: {best_accuracy:.1%}")


## Interpretation and limits
The winner is the model with the most correct predictions on these test flowers, not a general winner. The tree uses threshold rules; the forest combines bootstrap trees with feature subsampling; logistic regression fits linear class scores after scaling. A train/test gap can suggest overfitting but does not establish a cause.

Iris contains just 150 flowers and three species. A single small split has substantial sampling variability. Repeatedly inspecting this test set while choosing depth makes it part of exploration; it no longer provides an untouched final assessment. Broader conclusions need training-only cross-validation and a separate final test set. Model probabilities are not calibrated certainty.

The Streamlit app provides the same computation with interactive depth, a confusion matrix, a petal feature plot, individual errors and four measurement inputs.
